In [1]:
#Install requirements
%pip install -r "../requirements.txt"

Note: you may need to restart the kernel to use updated packages.


In [2]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix, 
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    ConfusionMatrixDisplay)

import scipy
from scipy.stats import pearsonr, uniform, loguniform

import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 

In [3]:
import sys
import os

#Add the src to the path
sys.path.append(os.path.abspath(os.path.join('..')))

In [4]:
#Import the data-feature matrices generated in Task 1

if not os.path.exists("../data/processed"):
    print("Processed Data Path does not exist. Expected in `../data/processed`. Run first data_exploration.ipynb")
    exit

#Load Training Data
X_train_metadata = pd.read_pickle('../data/processed/X_train_metadata.pkl')
X_train_cpg = pd.read_pickle('../data/processed/X_train_cpg.pkl')
X_train_final = pd.read_pickle('../data/processed/X_train_final.pkl')
y_train = pd.read_pickle('../data/processed/y_train.pkl')

#Load Validation Data
X_val_metadata = pd.read_pickle('../data/processed/X_val_metadata.pkl')
X_val_cpg = pd.read_pickle('../data/processed/X_val_cpg.pkl')
X_val_final = pd.read_pickle('../data/processed/X_val_final.pkl')
y_val = pd.read_pickle('../data/processed/y_val.pkl')

print('Data for training and validation set loaded successfully')

Data for training and validation set loaded successfully


In [5]:
#Import the selected-feature set from task 3. mRMR selected feature set for k=50

import json

file_path = '../data/selected_features/selected_features.json'

if os.path.exists(file_path):
    with open(file_path, 'r') as f:
        loaded_data = json.load(f)
    
    # Extract the list with the features
    selected_mrmr_features = loaded_data["mrmr_features"]
    
    print(f"Successfully loaded mRMR selected features list")
    
else:
    print("Error: JSON file not found. Check your directory path!")

Successfully loaded mRMR selected features list


In [6]:
#For hyperparameters tuning we are going to use the RandomizedSearchCV from scikit-learn
from sklearn.linear_model import ElasticNet, BayesianRidge
from sklearn.svm import SVR

Xtrain_set = X_train_cpg[selected_mrmr_features]
Xval_set = X_val_cpg[selected_mrmr_features]

#For tuning I will need the full development dataset, give cross validation more data to learn from
X_dev = pd.concat([Xtrain_set, Xval_set])
y_dev = pd.concat([y_train, y_val])

print(f"The training and validation sets for selected features have been combined. Size of the generated development dataset: {X_dev.shape}")

The training and validation sets for selected features have been combined. Size of the generated development dataset: (456, 50)


In [11]:
#For the ElasticNet model
#Define the regressor
elasticnet_reg = ElasticNet(random_state=42)

#Define the hyperparameteres search space
elasticnet_hyp = {'alpha': loguniform(0.001, 10), 'l1_ratio':uniform(0.1, 0.9)}

In [8]:
#For the SVR model
svr_reg = SVR()

#Define the hyperparameteres search space
svr_hyp = {'C':loguniform(0.1, 500), 'epsilon': [0.01, 0.1, 0.5, 1.0], 'kernel':['rbf', 'linear']}

In [9]:
#For the BayesianRidge model 
bayesianridge_reg = BayesianRidge()

#Define the hyperparameteres search space
bayesianridge_hyp = {'alpha_1':loguniform(1e-7,1e-3), 'alpha_2':loguniform(1e-7,1e-3), 'lambda_1':loguniform(1e-7,1e-3), 'lambda_2':loguniform(1e-7,1e-3)}

In [ ]:
#Apply the RandomizedSearchCv to the 3 models

from sklearn.model_selection import RandomizedSearchCV
    
#Dictionary with the models and their estimators and hyperparameter dictionary spaces
models = {
    'ElasticNet' : (elasticnet_reg, elasticnet_hyp),
    'SVR' : (svr_reg, svr_hyp),
    'BayesianRidge' : (bayesianridge_reg, bayesianridge_hyp)
}

#Initialize an emplty dictionary to save the best estimators
results_tuning = {}

for name, (model, parameters) in models.items():
    clf = RandomizedSearchCV(
        estimator=model,
        param_distributions=parameters,
        n_iter=40, #40 random iterations(trials) per model
        cv=5, #5-fold cross validation
        scoring='neg_root_mean_squared_error', #objective for minimization
        random_state=42
    )
    clf.fit(X_dev, y_dev)

    results_tuning[name]=clf

    print(f"Best RMSE for {name}: {-clf.best_score_:.4f}")
    print(f"Best Params: {clf.best_params_}\n")


Best RMSE for ElasticNet: 5.1771
Best Params: {'alpha': np.float64(0.2801635158716261), 'l1_ratio': np.float64(0.22554447458683766)}

Best RMSE for SVR: 5.1920
Best Params: {'C': np.float64(0.11916299962955149), 'epsilon': 0.1, 'kernel': 'linear'}

Best RMSE for BayesianRidge: 5.2436
Best Params: {'alpha_1': np.float64(3.798214508453255e-07), 'alpha_2': np.float64(9.074256288983855e-06), 'lambda_1': np.float64(0.0008761971101023686), 'lambda_2': np.float64(9.294394155644988e-07)}



In [14]:
#Save the results from tuning
tuning_results = []

for name, clf in results_tuning.items(): 
    params = clf.best_params_

    #Cleaning
    clean_params = {
        k: (round(float(v), 4) if isinstance(v, (float, np.float64, np.int64)) else v) 
        for k, v in params.items()
    }

    tuning_results.append({
        'Model': name,
        'Best Mean CV RMSE': round(-clf.best_score_,4),
        'Best Parameters': str(clean_params).replace("'", "")
    })

os.makedirs('../data/tuning', exist_ok=True)

#Summary table
summary_df = pd.DataFrame(tuning_results)
summary_df.to_csv('../data/tuning/Task4_Tuning_Summary.csv', index=False)

print(summary_df)

           Model  Best Mean CV RMSE  \
0     ElasticNet             5.1771   
1            SVR             5.1920   
2  BayesianRidge             5.2436   

                                     Best Parameters  
0                  {alpha: 0.2802, l1_ratio: 0.2255}  
1          {C: 0.1192, epsilon: 0.1, kernel: linear}  
2  {alpha_1: 0.0, alpha_2: 0.0, lambda_1: 0.0009,...  


In [16]:
#RandomizedSearchCV by default after finding the best hyperparameters perform refit of the estimator on the whole dataset using those hyperparameters (refit=True #default setting)
#Save the refitted tuned estimators  
import joblib

os.makedirs('../models/Tuned_estimators', exist_ok=True)

tuned_elasticnet = results_tuning['ElasticNet'].best_estimator_
tuned_svr = results_tuning['SVR'].best_estimator_
tuned_bayesianridge = results_tuning['BayesianRidge'].best_estimator_

joblib.dump(tuned_elasticnet, '../models/Tuned_estimators/tuned_elastinet.joblib' )
joblib.dump(tuned_svr, '../models/Tuned_estimators/tuned_svr.joblib' )
joblib.dump(tuned_bayesianridge, '../models/Tuned_estimators/tuned_bayesianridge.joblib' )

print(f"Tuned estimators saved")

Tuned estimators saved
